In [2]:
import os 
from glob import glob
import pandas as pd

In [3]:
# 특정 경로에서 파일의 목록을 가져오는 기능 
os.listdir("./review")

['1-1.여성의류(196).json',
 '1-1.여성의류(197).json',
 '1-1.여성의류(198).json',
 '1-1.여성의류(199).json',
 '1-1.여성의류(200).json',
 '1-1.여성의류(201).json',
 '1-1.여성의류(202).json',
 '1-1.여성의류(203).json',
 '1-1.여성의류(204).json',
 '1-1.여성의류(205).json',
 '1-1.여성의류(206).json',
 '1-1.여성의류(207).json',
 '1-1.여성의류(208).json',
 '1-1.여성의류(209).json']

In [4]:
# glob 라이브러리 사용
# 장점 : 파일의 경로와 파일 명이 동시에 출력 
#       특정 확장자만 목록을 불러올수 있다. 
json_list = glob("./review/*.json")

In [5]:
# json_list를 이용하여 여러 파일들을 하나의 데이터프레임으로 결합 

# 비어있는 데이터프레임을 생성 
total_df = pd.DataFrame()

for file_path in json_list:
    # print(file_path)
    # break
    df = pd.read_json(file_path)
    # df를 total_df에 단순 행 결합 
    total_df = pd.concat( [total_df, df], axis = 0 )
total_df.reset_index(drop = True, inplace=True)
total_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1423 entries, 0 to 1422
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            1423 non-null   int64  
 1   RawText          1423 non-null   str    
 2   Source           1423 non-null   str    
 3   Domain           1423 non-null   str    
 4   MainCategory     1423 non-null   str    
 5   ProductName      1423 non-null   str    
 6   Syllable         1423 non-null   int64  
 7   Word             1423 non-null   int64  
 8   GeneralPolarity  1418 non-null   float64
 9   Aspects          1423 non-null   object 
dtypes: float64(1), int64(3), object(1), str(5)
memory usage: 111.3+ KB


In [6]:
total_df.head(2)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1.0,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."
1,1024477,드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드...,SNS,패션,여성의류,OO 아** 구스코트,464,105,1.0,"[{'Aspect': '사이즈', 'SentimentText': '저는 블랙90 사..."


In [7]:
aspect_df = pd.DataFrame(sum(total_df['Aspects'], []))

In [8]:
aspect_df.head()

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0


In [9]:
# 결측치가 존재하는가?
aspect_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10962 entries, 0 to 10961
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Aspect             10962 non-null  str  
 1   SentimentText      10962 non-null  str  
 2   SentimentWord      10962 non-null  str  
 3   SentimentPolarity  10962 non-null  str  
dtypes: str(4)
memory usage: 342.7 KB


In [10]:
aspect_df = aspect_df.map(lambda x : x.strip())

In [11]:
(aspect_df == '').sum()

Aspect               0
SentimentText        0
SentimentWord        0
SentimentPolarity    0
dtype: int64

In [12]:
# 종속 변수들의 데이터의 빈도수를 확인 
aspect_df['Aspect'].value_counts()

Aspect
디자인     1676
기능      1212
소재      1151
활용성      917
색상       855
핏        753
가격       679
사이즈      642
착용감      633
길이       496
품질       468
두께       353
신축성      277
무게       271
촉감       252
제품구성     169
마감       142
냄새        16
Name: count, dtype: int64

In [13]:
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9664
-1    1005
0      293
Name: count, dtype: int64

In [14]:
# 독립 변수에서 중복된 데이터가 존재하면 제거 
aspect_df.drop_duplicates('SentimentText', inplace=True)

In [15]:
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     9183
-1     994
0      290
Name: count, dtype: int64

In [16]:
# 인덱스 초기화
aspect_df.reset_index(drop=True, inplace=True)

In [17]:
# 독립 변수를 토큰화 -> 벡터화 
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
# 토큰화 과정에서 특정 품사들만 사용
# 명사, 동사, 형용사, 부사

okt = Okt()
allow_pos = ['Noun', 'Verb', 'Adjective', 'Adverb']

def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if (pos in allow_pos) & (len(word) >= 2):
            result.append(word)
    return result

tokenize(
    aspect_df.loc[0, 'SentimentText']
)

['기본', '스타일', '입은', '보면', '깔끔하게', '저렴해', '보이지', '않는', '디자인']

In [19]:
vec = TfidfVectorizer(
    tokenizer= tokenize, 
    ngram_range= (1, 2), 
    min_df= 3, 
    max_df = 0.8, 
    max_features= 3000
)

In [20]:
aspect_df.columns

Index(['Aspect', 'SentimentText', 'SentimentWord', 'SentimentPolarity'], dtype='str')

In [21]:
X = aspect_df['SentimentText'].values
y1 = aspect_df['Aspect'].values
y2 = aspect_df['SentimentPolarity'].values

In [22]:
X_vec = vec.fit_transform(X)

c:\study\multicampus_practice\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [23]:
X_vec.shape

(10467, 3000)

In [24]:

from sklearn.preprocessing import LabelEncoder

In [25]:
le1 = LabelEncoder()
y1_le = le1.fit_transform(y1) 
le2 = LabelEncoder()
y2_le = le2.fit_transform(y2)

In [26]:
from sklearn.svm import LinearSVC

In [27]:
svc1 = LinearSVC(class_weight='balanced', random_state=42)
svc2 = LinearSVC(class_weight='balanced', random_state=42)